# Scratch 2 — Crypto Markets: Triage & Connector Build

End-to-end test of the pipeline on Polymarket crypto markets:
1. Build the inventory DataFrame via `inventory_crypto_markets`
2. Triage the first 20 rows with `triage_dataframe_incremental_async`
3. Filter out rows whose plans only use standard OHLC connectors
4. Run `build_connectors_async` on the remaining non-standard plans

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parents[0]))

import pandas as pd
pd.set_option('display.max_colwidth', None)

import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

from market_inventory.universe import CoinUniverse, ProjectUniverse
from market_inventory.inventory import inventory_crypto_markets
from market_inventory.polymarket_clients import GammaClient

coin_universe = CoinUniverse.from_json(Path("/workspaces/Agents_v1/coins_universe.json"))
project_universe = ProjectUniverse.from_json(Path("/workspaces/Agents_v1/projects_universe.json"))
gamma = GammaClient()

print("Setup OK")

## 1. Build Inventory

In [ ]:
df = inventory_crypto_markets(
    gamma=gamma,
    coin_universe=coin_universe,
    project_universe=project_universe,
    limit_events=500,
)
print(f"Inventory rows: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(5)

## 2. Filter Out Standard-Exchange Rows & Triage First 20

Drop rows whose `resolution_source` is one of the 7 standard OHLC exchanges
before triaging — no point spending agent calls on markets that resolve via
connectors we already have.

In [ ]:
from pipeline.historical_triage_runner import (
    triage_dataframe_incremental_async,
    load_triage_cache,
    save_triage_cache,
    diff_inventory_vs_cache,
    DIFF_COLUMNS,
    DEFAULT_CACHE_PATH,
)

# Standard exchange names as they appear in the resolution_source column
STANDARD_EXCHANGES = {
    "coingecko", "binance", "coinbase", "kraken", "bitstamp", "okx", "bybit",
}

# Filter out rows whose resolution_source is a standard exchange
mask_standard = df["resolution_source"].astype(str).str.lower().str.strip().isin(STANDARD_EXCHANGES)
df_non_standard = df[~mask_standard].reset_index(drop=True)

print(f"Total inventory rows   : {len(df)}")
print(f"Standard-exchange rows : {mask_standard.sum()}  (skipped)")
print(f"Non-standard rows      : {len(df_non_standard)}")

CACHE_PATH = Path("/tmp/scratch2_triage_cache.parquet")
CACHE_PATH.unlink(missing_ok=True)  # start clean

print(f"\nCache path: {CACHE_PATH}")
print(f"Rows to triage: first 20 of {len(df_non_standard)}")

In [ ]:
# Triage the first 20 non-standard rows
df_first20 = df_non_standard.head(20).copy().reset_index(drop=True)

df_triaged = await triage_dataframe_incremental_async(
    df_first20,
    cache_path=CACHE_PATH,
    max_rows=20,
    max_concurrency=10,
    row_timeout_s=120.0,
    total_timeout_s=600.0,
    log_every=5,
)

print(f"\nTriaged rows: {len(df_triaged)}")
print(f"Cache size: {CACHE_PATH.stat().st_size:,} bytes")
df_triaged[["market", "resolution_source", "historical_relevance", "data_feasibility", "paywall_risk", "triage_error"]].head(20)

## 3. Filter Out Standard Connectors & Build Custom Connectors

Skip any rows whose triage plans only reference standard OHLC connectors
(coingecko, binance, coinbase, kraken, bitstamp, okx, bybit).

In [ ]:
import json

# Standard connector function names to skip
STANDARD_CONNECTOR_FUNCTIONS = {
    "fetch_coingecko_ohlcv",
    "fetch_binance_ohlcv",
    "fetch_coinbase_ohlcv",
    "fetch_kraken_ohlcv",
    "fetch_bitstamp_ohlcv",
    "fetch_okx_ohlcv",
    "fetch_bybit_ohlcv",
}


def has_non_standard_plans(plans_json):
    """Return True if the row has at least one plan that is NOT a standard OHLC connector."""
    if not plans_json or (isinstance(plans_json, float) and pd.isna(plans_json)):
        return False
    try:
        plans = json.loads(plans_json)
    except Exception:
        return False
    if not plans:
        return False
    # Check if ANY plan has a non-standard connector function name
    for p in plans:
        fn_name = p.get("connector_function_name", "")
        if fn_name not in STANDARD_CONNECTOR_FUNCTIONS:
            return True
    return False


def filter_standard_plans(plans_json):
    """Return a new JSON string with standard OHLC connector plans removed."""
    if not plans_json or (isinstance(plans_json, float) and pd.isna(plans_json)):
        return plans_json
    try:
        plans = json.loads(plans_json)
    except Exception:
        return plans_json
    filtered = [
        p for p in plans
        if p.get("connector_function_name", "") not in STANDARD_CONNECTOR_FUNCTIONS
    ]
    return json.dumps(filtered, ensure_ascii=False)


# Filter: keep only rows that have at least one non-standard plan
mask = df_triaged["triage_plans_json"].apply(has_non_standard_plans)
df_for_build = df_triaged[mask].copy().reset_index(drop=True)

# Also strip out individual standard plans from the JSON so build_connectors_async
# doesn't try to build them
df_for_build["triage_plans_json"] = df_for_build["triage_plans_json"].apply(filter_standard_plans)

print(f"Rows with non-standard plans: {len(df_for_build)} / {len(df_triaged)}")
print(f"Rows skipped (standard-only or no plans): {len(df_triaged) - len(df_for_build)}")
df_for_build[["market", "historical_relevance", "data_feasibility", "triage_plans_json"]].head(10)

In [ ]:
from pipeline.connector_builder_runner import build_connectors_async

if len(df_for_build) == 0:
    print("No non-standard plans to build — all 20 rows use standard connectors only.")
    registry = {}
else:
    registry = await build_connectors_async(
        df_for_build,
        max_concurrency=5,
        skip_paywall=True,
        plan_timeout_s=120.0,
        total_timeout_s=600.0,
        log_every=5,
    )

print(f"\nRegistry size: {len(registry)} connectors built")
for key, c in registry.items():
    print(f"  {key}")
    print(f"    fn   : {c.connector_function_name}")
    print(f"    type : {c.connector_type}")
    print(f"    deps : {c.dependencies}")
    print(f"    notes: {c.notes}")
    print()

In [ ]:
# Display generated source code for each connector
for key, c in registry.items():
    print(f"{'='*80}")
    print(f"connector_key: {key}")
    print(f"function:      {c.connector_function_name}")
    print(f"{'='*80}")
    print("--- imports ---")
    for line in c.imports:
        print(line)
    print("\n--- source_code ---")
    print(c.source_code)
    print()